In [15]:
import pandas as pd
from pathlib import Path

RAW = Path('raw')
CLEAN = Path('cleaned')
CLEAN.mkdir(exist_ok=True)

geo = pd.read_csv(RAW/ 'olist_geolocation_dataset.csv', dtype={'geolocation_zip_code_prefix': str})

In [16]:
geo.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  str    
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), str(3)
memory usage: 38.2 MB


In [17]:
geo.isna().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [18]:
geo.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.545621,-46.639292,sao paulo,SP
1,01046,-23.546081,-46.644820,sao paulo,SP
2,01046,-23.546129,-46.642951,sao paulo,SP
3,01041,-23.544392,-46.639499,sao paulo,SP
4,01035,-23.541578,-46.641607,sao paulo,SP


In [19]:
geo.duplicated().sum()

np.int64(261831)

In [20]:
print(geo['geolocation_zip_code_prefix'].str.len().value_counts())
print(geo['geolocation_zip_code_prefix'].nunique())
print(geo.drop_duplicates().shape)
print(geo[['geolocation_lat', 'geolocation_lng']].describe())
print(geo['geolocation_state'].nunique())
print(geo['geolocation_city'].nunique())

geolocation_zip_code_prefix
5    1000163
Name: count, dtype: int64
19015
(738332, 5)
       geolocation_lat  geolocation_lng
count     1.000163e+06     1.000163e+06
mean     -2.117615e+01    -4.639054e+01
std       5.715866e+00     4.269748e+00
min      -3.660537e+01    -1.014668e+02
25%      -2.360355e+01    -4.857317e+01
50%      -2.291938e+01    -4.663788e+01
75%      -1.997962e+01    -4.376771e+01
max       4.506593e+01     1.211054e+02
27
8011


In [21]:
geo = geo.drop_duplicates().reset_index(drop=True)
print(geo.shape)                                    # (738332, 5)

in_brazil = geo['geolocation_lat'].between(-34, 5.5) & geo['geolocation_lng'].between(-74, -28)
print((~in_brazil).sum())                           # rows outside Brazil
print(geo.loc[~in_brazil, 'geolocation_state'].value_counts().head(10))
print(geo.loc[~in_brazil].head(10).to_string())

(738332, 5)
27
geolocation_state
RJ    5
PA    5
BA    4
RS    3
ES    2
MG    2
PR    2
SP    1
AL    1
PB    1
Name: count, dtype: int64
       geolocation_zip_code_prefix  geolocation_lat  geolocation_lng         geolocation_city geolocation_state
272046                       18243        28.008978       -15.536867  bom retiro da esperanca                SP
355591                       28165        41.614052        -8.411675      vila nova de campos                RJ
355603                       28155       -34.586422       -58.732101              santa maria                RJ
355694                       28155        42.439286        13.820214              santa maria                RJ
356233                       28333        38.381672        -6.328200                   raposo                RJ
358103                       28595        43.684961        -7.411080                  portela                RJ
374435                       29654        29.409252       -98.484121   santo 

In [22]:
odd = geo.loc[geo['geolocation_city'].str.contains(r"[^a-z \-']", regex=True), 'geolocation_city']
print(odd.shape[0])
print(odd.nunique())
print(odd.drop_duplicates().head(30).to_string())

63457
2104
5                      são paulo
29567                    jundiaí
44552            taboão da serra
49274                   sãopaulo
63566                 sa£o paulo
94733                carapicuíba
100348       santana de parnaíba
107421     são lourenço da serra
109932                embu-guaçu
109935                embu guaçu
117892                     arujá
118785                 mairiporã
119672                      mauá
119989                jordanésia
131881                       poá
139293               salesópolis
139613               santo andré
147841            ribeirão pires
149060        são caetano do sul
150873     são bernardo do campo
162315               são vicente
163645                   guarujá
165744                   cubatão
166502             são sebastião
169154    são francisco da praia
169200                  mongaguá
169278                  itanhaém
169380                   peruíbe
172846                    juquiá
173015                  cananéia

In [23]:
geo['coord_outside_brazil'] = ~in_brazil

c = geo['geolocation_city'].str.replace('´', "'", regex=False)
c = c.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('ascii').str.strip()
geo['geolocation_city'] = c

print(geo['coord_outside_brazil'].sum())            # 27
print(geo['geolocation_city'].nunique())            # was 8,011

27
5967


In [24]:
g = geo.groupby('geolocation_zip_code_prefix')
print((g['geolocation_state'].nunique() > 1).sum())   # zips with more than one state
print((g['geolocation_city'].nunique() > 1).sum())    # zips with more than one city name
print(g.size().describe())                            # points per zip

8
549
count    19015.000000
mean        38.828925
std         50.875735
min          1.000000
25%          8.000000
50%         23.000000
75%         49.000000
max        779.000000
dtype: float64


In [25]:
multi_state = g['geolocation_state'].nunique()
bad_zips = multi_state[multi_state > 1].index
print(geo[geo['geolocation_zip_code_prefix'].isin(bad_zips)]
      .groupby(['geolocation_zip_code_prefix', 'geolocation_state']).size().to_string())

multi_city = g['geolocation_city'].nunique()
ex = multi_city[multi_city > 1].index[:6]
print(geo[geo['geolocation_zip_code_prefix'].isin(ex)]
      .groupby(['geolocation_zip_code_prefix', 'geolocation_city']).size().to_string())

geolocation_zip_code_prefix  geolocation_state
02116                        RN                     1
                             SP                    10
04011                        AC                     1
                             SP                    69
21550                        AC                     1
                             RJ                   144
23056                        AC                     1
                             RJ                    30
72915                        DF                     1
                             GO                     9
78557                        MT                    75
                             RO                     1
79750                        MS                   151
                             RS                     1
80630                        PR                    76
                             SC                     1
geolocation_zip_code_prefix  geolocation_city     
01307                        sao berna

In [26]:
zkey = 'geolocation_zip_code_prefix'
mode = lambda s: s.mode().iloc[0]

state = geo.groupby(zkey)['geolocation_state'].agg(mode)
city  = geo.groupby(zkey)['geolocation_city'].agg(mode)

pts = geo.loc[~geo['coord_outside_brazil']].copy()
pts = pts[pts['geolocation_state'] == pts[zkey].map(state)]
coords = pts.groupby(zkey)[['geolocation_lat', 'geolocation_lng']].median()

points_count = geo.groupby(zkey).size().rename('points_count')

geo_zip = pd.concat([city, state, coords, points_count], axis=1).reset_index()

print(geo_zip.shape)                       # expect (19015, 6)
print(geo_zip[zkey].is_unique)             # True
print(geo_zip.isna().sum())                # zips left without coordinates
print(geo_zip['geolocation_city'].nunique())
print(geo_zip.head())

(19015, 6)
True
geolocation_zip_code_prefix    0
geolocation_city               0
geolocation_state              0
geolocation_lat                4
geolocation_lng                4
points_count                   0
dtype: int64
5771
  geolocation_zip_code_prefix geolocation_city geolocation_state  \
0                       01001        sao paulo                SP   
1                       01002        sao paulo                SP   
2                       01003        sao paulo                SP   
3                       01004        sao paulo                SP   
4                       01005        sao paulo                SP   

   geolocation_lat  geolocation_lng  points_count  
0       -23.549951       -46.634027            11  
1       -23.548228       -46.635247             6  
2       -23.548977       -46.635313            11  
3       -23.549550       -46.634771            14  
4       -23.549763       -46.636100            13  


In [27]:
print(geo_zip[geo_zip['geolocation_lat'].isna()].to_string())

cust = pd.read_csv(CLEAN/'customers_clean.csv', dtype={'customer_zip_code_prefix': str},
                   usecols=['customer_zip_code_prefix'])
sell = pd.read_csv(CLEAN/'sellers_clean.csv', dtype={'seller_zip_code_prefix': str},
                   usecols=['seller_zip_code_prefix'])

has = geo_zip.loc[geo_zip['geolocation_lat'].notna(), zkey]

print((~cust['customer_zip_code_prefix'].isin(has)).sum())                  # customer rows with no coordinates
print(cust.loc[~cust['customer_zip_code_prefix'].isin(has), 'customer_zip_code_prefix'].nunique())  # distinct zips behind them
print((~sell['seller_zip_code_prefix'].isin(has)).sum())                    # seller rows with no coordinates

      geolocation_zip_code_prefix         geolocation_city geolocation_state  geolocation_lat  geolocation_lng  points_count
6077                        18243  bom retiro da esperanca                SP              NaN              NaN             1
15781                       78131            varzea grande                MT              NaN              NaN             1
16419                       83252       ilha dos valadares                PR              NaN              NaN             1
18309                       95130      santa lucia do piai                RS              NaN              NaN             1
279
158
7


In [28]:
geo_zip.to_csv(CLEAN/'geolocation_clean.csv', index=False)

check = pd.read_csv(CLEAN/'geolocation_clean.csv', dtype={'geolocation_zip_code_prefix': str})
print(check.shape)                                          # (19015, 6)
print(check['geolocation_zip_code_prefix'].is_unique)       # True
print(check['geolocation_zip_code_prefix'].str.len().value_counts())   # all 5
print(check['geolocation_lat'].isna().sum())                # 4
print(check['geolocation_lat'].between(-34, 5.5).all())     # NaN rows give False; use the next line instead
print(check[['geolocation_lat', 'geolocation_lng']].describe())

(19015, 6)
True
geolocation_zip_code_prefix
5    19015
Name: count, dtype: int64
4
False
       geolocation_lat  geolocation_lng
count     19011.000000     19011.000000
mean        -19.078966       -46.077180
std           7.240744         5.198294
min         -33.690729       -72.909444
25%         -23.564539       -49.008455
50%         -22.425826       -46.632757
75%         -15.617623       -43.253402
max           4.478429       -32.412876
